# BikeToDrive SDM vullen

Dit notebook voert stap 5 uit:

1. **reset** van alle tabellen in het Source Data Model  
2. **inladen** van data uit de 5 operationele SQLite-bronnen naar het SDM  
3. **controle** van aantallen en foreign keys

## Gekozen inlaadstrategie
Hier wordt een **full refresh / truncate-and-load** strategie gebruikt:

- eerst worden alle SDM-tabellen leeggemaakt
- daarna wordt elke bron **1-op-1** ingeladen in de bijbehorende SDM-tabellen
- de laadvolgorde volgt de foreign keys: eerst stamtabellen, daarna transactietabellen

Dat past hier goed, omdat:
- het SDM volledig opnieuw opgebouwd mag worden
- de brondata relatief klein is
- de SDM-tabellen bron-specifiek zijn benoemd, waardoor records uit verschillende databases elkaar niet overschrijven


In [1]:

from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.home() / "Downloads"

SOURCE_DBS = {
    "accessoireverkoop": BASE_DIR / "BikeToDrive_1_Accessoireverkoop.db",
    "fietsverkoop": BASE_DIR / "BikeToDrive_2_Fietsverkoop.db",
    "onderhoud": BASE_DIR / "BikeToDrive_3_Onderhoud.db",
    "accessoire_inkoop": BASE_DIR / "BikeToDrive_4_Accessoire_Inkoop.db",
    "fiets_inkoop": BASE_DIR / "BikeToDrive_5_Fiets_Inkoop.db",
}

SDM_DB = BASE_DIR / "BikeToDrive_SDM.db"

TABLE_MAPPING = {
    "accessoireverkoop": [
        ("Filiaal", "Accessoire_Verkoop_Filiaal"),
        ("Klant", "Accessoire_Verkoop_Klant"),
        ("Leverancier", "Accessoire_Verkoop_Leverancier"),
        ("Monteur", "Accessoire_Verkoop_Monteur"),
        ("Accessoire", "Accessoire_Verkoop_Accessoire"),
        ("Accessoire_Verkoop", "Accessoire_Verkoop"),
    ],
    "fietsverkoop": [
        ("Filiaal", "Fiets_Verkoop_Filiaal"),
        ("Klant", "Fiets_Verkoop_Klant"),
        ("Fabrikant", "Fiets_Verkoop_Fabrikant"),
        ("Monteur", "Fiets_Verkoop_Monteur"),
        ("Fiets", "Fiets_Verkoop_Fiets"),
        ("Fiets_Verkoop", "Fiets_Verkoop"),
    ],
    "onderhoud": [
        ("Filiaal", "Onderhoud_Filiaal"),
        ("Fabrikant", "Onderhoud_Fabrikant"),
        ("Fiets", "Onderhoud_Fiets"),
        ("Monteur", "Onderhoud_Monteur"),
        ("Onderhoud", "Onderhoud"),
    ],
    "accessoire_inkoop": [
        ("Leverancier", "Accessoire_Inkoop_Leverancier"),
        ("Accessoire", "Accessoire_Inkoop_Accessoire"),
        ("Accessoire_Inkoop", "Accessoire_Inkoop"),
    ],
    "fiets_inkoop": [
        ("Fabrikant", "Fiets_Inkoop_Fabrikant"),
        ("Fiets", "Fiets_Inkoop_Fiets"),
        ("Fiets_Inkoop", "Fiets_Inkoop"),
    ],
}

for label, path in SOURCE_DBS.items():
    if not path.exists():
        raise FileNotFoundError(f"Bronbestand ontbreekt: {path}")

if not SDM_DB.exists():
    raise FileNotFoundError(f"SDM-bestand ontbreekt: {SDM_DB}")

print("Bestanden gevonden.")
print("SDM:", SDM_DB)
for label, path in SOURCE_DBS.items():
    print(f"- {label}: {path.name}")

def get_user_tables(conn: sqlite3.Connection) -> list[str]:
    rows = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()
    return [row[0] for row in rows]

def reset_sdm(sdm_path: Path) -> None:
    with sqlite3.connect(sdm_path) as conn:
        conn.execute("PRAGMA foreign_keys = OFF")
        tables = get_user_tables(conn)
        for table in tables:
            conn.execute(f'DELETE FROM "{table}"')
        conn.commit()
        conn.execute("PRAGMA foreign_keys = ON")
    print(f"{len(tables)} tabellen leeggemaakt.")

def fetch_all_rows(conn: sqlite3.Connection, table_name: str):
    conn.row_factory = sqlite3.Row
    return conn.execute(f'SELECT * FROM "{table_name}"').fetchall()

def load_table(source_conn: sqlite3.Connection, target_conn: sqlite3.Connection, source_table: str, target_table: str) -> int:
    rows = fetch_all_rows(source_conn, source_table)
    if not rows:
        return 0

    columns = list(rows[0].keys())
    column_sql = ", ".join([f'"{col}"' for col in columns])
    placeholders = ", ".join(["?"] * len(columns))

    target_conn.executemany(
        f'INSERT INTO "{target_table}" ({column_sql}) VALUES ({placeholders})',
        [tuple(row[col] for col in columns) for row in rows]
    )
    return len(rows)

def load_all_sources(sdm_path: Path, source_dbs: dict, table_mapping: dict) -> pd.DataFrame:
    results = []

    with sqlite3.connect(sdm_path) as target_conn:
        target_conn.execute("PRAGMA foreign_keys = OFF")

        for source_label, mappings in table_mapping.items():
            with sqlite3.connect(source_dbs[source_label]) as source_conn:
                for source_table, target_table in mappings:
                    loaded_rows = load_table(source_conn, target_conn, source_table, target_table)
                    results.append({
                        "bron": source_label,
                        "bron_tabel": source_table,
                        "sdm_tabel": target_table,
                        "geladen_rijen": loaded_rows
                    })

        target_conn.commit()
        target_conn.execute("PRAGMA foreign_keys = ON")

    return pd.DataFrame(results)

def foreign_key_issues(sdm_path: Path):
    with sqlite3.connect(sdm_path) as conn:
        return conn.execute("PRAGMA foreign_key_check").fetchall()

def row_counts(sdm_path: Path) -> pd.DataFrame:
    with sqlite3.connect(sdm_path) as conn:
        tables = get_user_tables(conn)
        data = []
        for table in tables:
            count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
            data.append({"tabel": table, "aantal_rijen": count})
    return pd.DataFrame(data).sort_values(["tabel"]).reset_index(drop=True)


Bestanden gevonden.
SDM: /Users/cmok/Downloads/BikeToDrive_SDM.db
- accessoireverkoop: BikeToDrive_1_Accessoireverkoop.db
- fietsverkoop: BikeToDrive_2_Fietsverkoop.db
- onderhoud: BikeToDrive_3_Onderhoud.db
- accessoire_inkoop: BikeToDrive_4_Accessoire_Inkoop.db
- fiets_inkoop: BikeToDrive_5_Fiets_Inkoop.db


## 1. Reset-knop: alle SDM-tabellen leegmaken

In [2]:
reset_sdm(SDM_DB)

23 tabellen leeggemaakt.


## 2. Data uit alle 5 bronbestanden overzetten naar het SDM

De laadvolgorde is per bron zo gekozen dat eerst de tabellen zonder afhankelijke foreign keys worden geladen en daarna de transactietabellen.


In [3]:
load_result = load_all_sources(SDM_DB, SOURCE_DBS, TABLE_MAPPING)
load_result

,bron,bron_tabel,sdm_tabel,geladen_rijen
0,accessoireverkoop,Filiaal,Accessoire_Verkoop_Filiaal,4
1,accessoireverkoop,Klant,Accessoire_Verkoop_Klant,20
2,accessoireverkoop,Leverancier,Accessoire_Verkoop_Leverancier,5
3,accessoireverkoop,Monteur,Accessoire_Verkoop_Monteur,10
4,accessoireverkoop,Accessoire,Accessoire_Verkoop_Accessoire,10
5,accessoireverkoop,Accessoire_Verkoop,Accessoire_Verkoop,100
6,fietsverkoop,Filiaal,Fiets_Verkoop_Filiaal,4
7,fietsverkoop,Klant,Fiets_Verkoop_Klant,25
8,fietsverkoop,Fabrikant,Fiets_Verkoop_Fabrikant,10
9,fietsverkoop,Monteur,Fiets_Verkoop_Monteur,10


## 3. Controle van de inhoud

In [4]:
counts_df = row_counts(SDM_DB)
counts_df

,tabel,aantal_rijen
0,Accessoire_Inkoop,50
1,Accessoire_Inkoop_Accessoire,13
2,Accessoire_Inkoop_Leverancier,5
3,Accessoire_Verkoop,100
4,Accessoire_Verkoop_Accessoire,10
5,Accessoire_Verkoop_Filiaal,4
6,Accessoire_Verkoop_Klant,20
7,Accessoire_Verkoop_Leverancier,5
8,Accessoire_Verkoop_Monteur,10
9,Fiets_Inkoop,100


## 4. Foreign key controle

In [5]:
fk_errors = foreign_key_issues(SDM_DB)
print('Aantal foreign key issues:', len(fk_errors))
fk_errors[:10]

Aantal foreign key issues: 0


[]

## Conclusie

Als `Aantal foreign key issues: 0` wordt getoond, dan is het SDM correct gevuld en zijn de verwijzingen geldig.
